# Simple LangChain RAG using Apache Solr

This notebook keeps the original LangChain RAG flow and replaces ChromaDB with Apache Solr.

Solr runs as a separate local search server. The notebook sends document embeddings to Solr and uses Solr KNN search to retrieve the most similar chunks.


## Before running the notebook: check Solr on Windows

Open **PowerShell** in your Solr `bin` folder, for example:

```powershell
cd C:\solr-9.10.1\bin
```

Check Java, the Solr CLI, and the running server:

```powershell
java -version
.\solr.cmd version
.\solr.cmd status
```

If Solr is not running, start it:

```powershell
.\solr.cmd start -p 8983
```

Create the RAG core once:

```powershell
.\solr.cmd create -c rag_documents
```

Then open <http://localhost:8983/solr/>. If the Admin UI loads and `rag_documents` appears in the Core Selector, Solr is ready.

To stop it later:

```powershell
.\solr.cmd stop -p 8983
```


In [1]:
# ============================================================
# Cell 1 - Install Required Libraries
# ============================================================

%pip install -qU langchain langchain-community langchain-core
%pip install -qU langchain-text-splitters langchain-huggingface langchain-ollama
%pip install -qU sentence-transformers requests
%pip install -qU pypdf docx2txt beautifulsoup4 lxml unstructured


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Cell 2 - Import Required Libraries
# ============================================================

import hashlib
from pathlib import Path

import requests

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    UnstructuredHTMLLoader,
)
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All libraries imported successfully.")


C:\Users\AbidDileepJan\AppData\Local\Temp\ipykernel_9108\434677942.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
c:\Users\AbidDileepJan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.


In [3]:
# ============================================================
# Cell 3 - Project Configuration
# ============================================================

DATA_FOLDER = "documents"

SOLR_BASE_URL = "http://localhost:8983/solr"
SOLR_CORE = "rag_documents"
SOLR_URL = f"{SOLR_BASE_URL}/{SOLR_CORE}"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_DIMENSION = 384
LLM_MODEL = "llama3.2:1b"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
TOP_K = 4

# True removes old records from this RAG core before re-indexing.
CLEAR_EXISTING_INDEX = True

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

llm = ChatOllama(model=LLM_MODEL, temperature=0.3)

print("Configuration loaded successfully.")
print("Solr URL       :", SOLR_URL)
print("Embedding model:", EMBEDDING_MODEL)
print("LLM model      :", LLM_MODEL)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 677.61it/s]


Configuration loaded successfully.
Solr URL       : http://localhost:8983/solr/rag_documents
Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model      : llama3.2:1b


In [4]:
# ============================================================
# Cell 4 - Check Solr and Configure the Vector Schema
# ============================================================

def solr_get(path, **kwargs):
    response = requests.get(f"{SOLR_URL}{path}", timeout=30, **kwargs)
    response.raise_for_status()
    return response


def solr_post(path, **kwargs):
    response = requests.post(f"{SOLR_URL}{path}", timeout=60, **kwargs)
    response.raise_for_status()
    return response


# This fails with a clear message if Solr or the core is unavailable.
try:
    status = solr_get("/select", params={"q": "*:*", "rows": 0, "wt": "json"})
    print("Solr connection successful.")
    print("Documents currently stored:", status.json()["response"]["numFound"])
except requests.RequestException as error:
    raise RuntimeError(
        "Cannot connect to the Solr core. Start Solr and create it with: "
        ".\\solr.cmd create -c rag_documents"
    ) from error


def schema_item_exists(item_type, name):
    response = requests.get(
        f"{SOLR_URL}/schema/{item_type}/{name}",
        timeout=30,
    )
    return response.status_code == 200


if not schema_item_exists("fieldtypes", "knn_vector_384"):
    solr_post(
        "/schema",
        json={
            "add-field-type": {
                "name": "knn_vector_384",
                "class": "solr.DenseVectorField",
                "vectorDimension": VECTOR_DIMENSION,
                "similarityFunction": "cosine",
                "knnAlgorithm": "hnsw",
            }
        },
    )

fields = {
    "content": {"type": "text_general", "indexed": True, "stored": True},
    "source": {"type": "string", "indexed": True, "stored": True},
    "page": {"type": "string", "indexed": True, "stored": True},
    "vector": {"type": "knn_vector_384", "indexed": True, "stored": True},
}

for field_name, settings in fields.items():
    if not schema_item_exists("fields", field_name):
        solr_post(
            "/schema",
            json={"add-field": {"name": field_name, **settings}},
        )

print("Solr vector schema is ready.")


Solr connection successful.
Documents currently stored: 0
Solr vector schema is ready.


In [5]:
# ============================================================
# Cell 5 - Load Documents
# ============================================================

documents = []

loaders = {
    "*.pdf": PyPDFLoader,
    "*.docx": Docx2txtLoader,
    "*.html": UnstructuredHTMLLoader,
}

for pattern, loader_class in loaders.items():
    for file in Path(DATA_FOLDER).glob(pattern):
        print("Loading:", file.name)
        documents.extend(loader_class(str(file)).load())

if not documents:
    raise FileNotFoundError(
        f"No PDF, DOCX, or HTML files found in the '{DATA_FOLDER}' folder."
    )

print("Total document sections loaded:", len(documents))


Loading: Oil and gas production handbook ed3x0_web.pdf
Loading: US10655422.pdf
Loading: Oil_Gas_Drilling_Glossary.docx
Loading: hal-20251231.html
Total document sections loaded: 182


In [6]:
# ============================================================
# Cell 6 - Chunk the Documents
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunked_documents = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunked_documents))
print("First chunk source:", chunked_documents[0].metadata.get("source"))
print("First chunk preview:\n", chunked_documents[0].page_content[:500])


Total chunks created: 841
First chunk source: documents\Oil and gas production handbook ed3x0_web.pdf
First chunk preview:
 Oil and gas production handbook  
An introduction to oil and gas production, 
transport, refining and petrochemical 
industry
Håvard Devold


In [7]:
# ============================================================
# Cell 7 - Create Embeddings and Store Chunks in Solr
# ============================================================

if CLEAR_EXISTING_INDEX:
    solr_post("/update", params={"commit": "true"}, json={"delete": {"query": "*:*"}})

texts = [doc.page_content for doc in chunked_documents]
vectors = embedding_model.embed_documents(texts)

solr_documents = []

for chunk_number, (doc, vector) in enumerate(zip(chunked_documents, vectors)):
    source = str(doc.metadata.get("source", "Unknown"))
    page = str(doc.metadata.get("page", "N/A"))
    unique_text = f"{source}|{page}|{chunk_number}|{doc.page_content}"
    chunk_id = hashlib.sha256(unique_text.encode("utf-8")).hexdigest()

    solr_documents.append(
        {
            "id": chunk_id,
            "content": doc.page_content,
            "source": source,
            "page": page,
            "vector": vector,
        }
    )

# Send records in small batches.
BATCH_SIZE = 100
for start in range(0, len(solr_documents), BATCH_SIZE):
    batch = solr_documents[start : start + BATCH_SIZE]
    solr_post("/update", params={"commit": "true"}, json=batch)

count = solr_get(
    "/select",
    params={"q": "*:*", "rows": 0, "wt": "json"},
).json()["response"]["numFound"]

print("Chunks stored in Solr:", count)


Chunks stored in Solr: 841


In [9]:
# ============================================================
# Cell 8 - Create the Solr Retriever
# ============================================================

def search_solr(question):
    query_vector = embedding_model.embed_query(question)

    vector_text = (
        "["
        + ",".join(f"{value:.8f}" for value in query_vector)
        + "]"
    )

    response = solr_get(
        "/select",
        params={
            "q": f"{{!knn f=vector topK={TOP_K}}}{vector_text}",
            "fl": "content,source,page,score",
            "rows": TOP_K,
            "wt": "json",
        },
    ).json()

    documents = []

    for item in response["response"]["docs"]:
        content = item.get("content", "")
        source = item.get("source", "Unknown")
        page = item.get("page", "N/A")

        # Solr may return stored fields as lists
        if isinstance(content, list):
            content = "\n".join(
                str(value) for value in content
            )
        else:
            content = str(content)

        if isinstance(source, list):
            source = source[0] if source else "Unknown"

        if isinstance(page, list):
            page = page[0] if page else "N/A"

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": str(source),
                    "page": str(page),
                    "score": item.get("score"),
                },
            )
        )

    return documents


# Create the retriever
retriever = RunnableLambda(search_solr)

# Test the retriever
test_docs = retriever.invoke(
    "What is this document about?"
)

print("Retriever created successfully.")
print("Test chunks returned:", len(test_docs))

Retriever created successfully.
Test chunks returned: 4


In [10]:
# ============================================================
# Cell 9 - Prompt Template
# ============================================================

PROMPT_TEMPLATE = """
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

Instructions:
1. Use only the information from the context.
2. Do not make up or assume information.
3. If the answer is not found in the context, reply:
   "I could not find the answer in the provided documents."
4. Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

print("Prompt template created successfully.")


Prompt template created successfully.


In [11]:
# ============================================================
# Cell 10 - RAG Function with Conversation Memory and Sources
# ============================================================

chat_history = []


def ask_rag(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    history = ""
    for chat in chat_history[-3:]:
        history += f"User: {chat['question']}\n"
        history += f"Assistant: {chat['answer']}\n\n"

    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    if history:
        prompt = f"Previous Conversation:\n{history}\n\n{prompt}"

    answer = llm.invoke(prompt).content
    chat_history.append({"question": question, "answer": answer})

    print("\n" + "=" * 70)
    print("Answer")
    print("=" * 70)
    print(answer)

    print("\n" + "=" * 70)
    print("Sources")
    print("=" * 70)

    seen = set()
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")
        key = (source, page)

        if key not in seen:
            print("File :", source)
            print("Page :", page)
            print("-" * 50)
            seen.add(key)


In [12]:
# ============================================================
# Cell 11 - Interactive Chat
# ============================================================

print("=" * 60)
print("RAG Chatbot - LangChain + Solr + Ollama")
print("Type 'exit' to stop")
print("=" * 60)

while True:
    question = input("\nAsk a question: ").strip()

    if question.lower() == "exit":
        print("Goodbye!")
        break

    if question:
        ask_rag(question)


RAG Chatbot - LangChain + Solr + Ollama
Type 'exit' to stop

Answer
Drilling mud is a specially formulated liquid or gas circulated down the drill string and back up the annulus during drilling, used to cool and lubricate the bit, remove cuttings, and control formation pressure.

Sources
File : documents\Oil and gas production handbook ed3x0_web.pdf
Page : 35
--------------------------------------------------
File : documents\Oil_Gas_Drilling_Glossary.docx
Page : N/A
--------------------------------------------------

Answer
I could not find the answer "escape" in the provided documents.

Sources
File : documents\hal-20251231.html
Page : N/A
--------------------------------------------------
File : documents\Oil and gas production handbook ed3x0_web.pdf
Page : 129
--------------------------------------------------
File : documents\Oil and gas production handbook ed3x0_web.pdf
Page : 21
--------------------------------------------------
File : documents\US10655422.pdf
Page : 13
--------